# PrivacyGuard FL — End-to-End Demo

This notebook walks through the full PrivacyGuard FL pipeline:
1. Load MNIST data and split into non-IID shards across 4 simulated clients
2. Train standard Federated Learning (FedAvg)
3. Train DP-Federated Learning (DP-SGD)
4. Run the Gradient Inversion Attack on unprotected and DP-protected gradients
5. Compare accuracy, privacy budget, and reconstruction quality

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import os
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

from src.data.pipeline import DataPipe
from src.core.federated import Client, FedServer, MNISTModel
from src.differential_privacy.dp_client import DPClient, DPFedServer
from src.attacks.gradient_inversion import GradientInversionAttack
from src.ui.visualization import (
    plot_training_history, plot_label_distribution,
    plot_attack_comparison, render_summary
)
import torch

print('Imports OK')

## 1. Data — Non-IID Split

We simulate 4 hospitals, each with 1500 samples concentrated on 2 digit classes (non-IID degree = 0.8).

In [ ]:
DEVICE = torch.device('cpu')
NUM_CLIENTS = 4
SAMPLES_PER_CLIENT = 1500
BATCH_SIZE = 32
ROUNDS = 15
LOCAL_EPOCHS = 2

pipe = DataPipe(
    num_clients=NUM_CLIENTS,
    samples_per_client=SAMPLES_PER_CLIENT,
    non_iid_degree=0.8,
    batch_size=BATCH_SIZE,
)
client_loaders, test_loader, client_indices = pipe.build_client_loaders()
train_set, _ = pipe.load_full_dataset()
dists = pipe.compute_label_distribution(train_set, client_indices)
plot_label_distribution(dists, save_path='output/label_distribution.png')
print(f'{NUM_CLIENTS} clients, {SAMPLES_PER_CLIENT} samples each')
print('Label distribution plot saved to output/label_distribution.png')

## 2. Standard Federated Learning

Train a model using FedAvg across all 4 clients for 15 rounds.

In [ ]:
print('--- Standard FL ---')
model = MNISTModel().to(DEVICE)
clients = [
    Client(i, MNISTModel(), loader, local_epochs=LOCAL_EPOCHS, device=DEVICE)
    for i, loader in enumerate(client_loaders)
]
server = FedServer(model, clients, test_loader, rounds=ROUNDS, device=DEVICE)
fl_history = server.fit()
fl_acc = fl_history['test_accuracy'][-1] * 100
print(f'Standard FL accuracy: {fl_acc:.1f}%')
torch.save(server.global_model.state_dict(), 'output/fl_model.pt')

## 3. Differential Privacy FL

Add DP-SGD to each client: per-sample gradient clipping (C=1.0) + Gaussian noise. Privacy budget ε=8.0, δ=1e-5.

In [ ]:
print('--- DP-FL ---')
dp_clients = [
    DPClient(
        i, MNISTModel(), loader,
        epsilon=8.0, delta=1e-5, clip_norm=1.0,
        learning_rate=0.05, local_epochs=LOCAL_EPOCHS,
        total_rounds=ROUNDS, device=DEVICE,
    )
    for i, loader in enumerate(client_loaders)
]
dp_server = DPFedServer(MNISTModel(), dp_clients, test_loader, rounds=ROUNDS, device=DEVICE)
dp_history = dp_server.fit()
dp_acc = dp_history['test_accuracy'][-1] * 100
dp_eps = dp_history['privacy_spent'][-1]
print(f'DP-FL accuracy: {dp_acc:.1f}%  (ε={dp_eps:.2f})')
torch.save(dp_server.global_model.state_dict(), 'output/dp_model.pt')
torch.save(dp_server.global_model.state_dict(), 'models/final_model.pt')

## 4. Training Curves

Side-by-side comparison of accuracy and loss over rounds for both models.

In [ ]:
plot_training_history(fl_history, dp_history, save_path='output/training_curves.png')
print('Training curves saved to output/training_curves.png')

# Also plot epsilon over rounds
import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dp_history['privacy_spent'], color='crimson', linewidth=2, marker='o')
ax.axhline(y=8.0, color='gray', linestyle='--', linewidth=1, label='Target ε (8.0)')
ax.set_xlabel('Round')
ax.set_ylabel('Privacy Budget ε')
ax.set_title('Privacy Budget Spent over Rounds')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/privacy_budget.png', dpi=120, bbox_inches='tight')
plt.close()
print('Privacy budget plot saved to output/privacy_budget.png')

## 5. Gradient Inversion Attack

Demonstrate iDLG reconstruction from unprotected and DP-protected gradients. The contrast between the two shows DP's privacy protection.

In [ ]:
print('--- Attack Demo ---')
pipe_attack = DataPipe(num_clients=NUM_CLIENTS, samples_per_client=SAMPLES_PER_CLIENT, non_iid_degree=0.8, batch_size=1)
attack_loaders, _, _ = pipe_attack.build_client_loaders()

model_atk = MNISTModel().to(DEVICE)
data, target = next(iter(attack_loaders[0]))
data, target = data.to(DEVICE), target.to(DEVICE)

model_atk.zero_grad()
out = model_atk(data)
loss = torch.nn.CrossEntropyLoss()(out, target)
loss.backward()
orig_w = {n: p.cpu().numpy().copy() for n, p in model_atk.state_dict().items()}
nd_grad = {n: p.grad.cpu().numpy().copy() for n, p in model_atk.named_parameters() if p.grad is not None}

# DP gradient
dp_atk = DPClient(0, MNISTModel(), attack_loaders[0], epsilon=8.0, local_epochs=1, total_rounds=1, device=DEVICE)
dp_atk.model.load_state_dict({n: torch.from_numpy(v).to(DEVICE) for n, v in orig_w.items()})
dp_grad = dp_atk.compute_dp_gradient(data, target)

print(f'True label: {target[0].item()}, DP σ={dp_atk.noise_multiplier:.4f}')

# Attack no-DP
att_nd = GradientInversionAttack(model_atk, orig_w, nd_grad, device=DEVICE)
recon_nd, _, _ = att_nd.attack(batch_size=1, steps=300, learning_rate=0.3, verbose=False)
mse_nd = GradientInversionAttack.evaluate_reconstruction_mse(data, recon_nd)
ssim_nd = GradientInversionAttack.evaluate_ssim(data, recon_nd)
print(f'No DP: MSE={mse_nd:.6f}  SSIM={ssim_nd:.4f}')

# Attack DP
att_dp = GradientInversionAttack(model_atk, orig_w, dp_grad, device=DEVICE)
recon_dp, _, _ = att_dp.attack(batch_size=1, steps=300, learning_rate=0.3, verbose=False)
mse_dp = GradientInversionAttack.evaluate_reconstruction_mse(data, recon_dp)
ssim_dp = GradientInversionAttack.evaluate_ssim(data, recon_dp)
print(f'With DP: MSE={mse_dp:.6f}  SSIM={ssim_dp:.4f}')
print(f'DP / No-DP MSE ratio: {mse_dp/(mse_nd+1e-8):.1f}x')

plot_attack_comparison(data, recon_nd, recon_dp, save_path='output/attack_comparison.png')
print('Attack comparison saved to output/attack_comparison.png')

## Summary

The privacy-utility tradeoff is visible:
- Standard FL: higher accuracy, no privacy protection
- DP-FL: slightly lower accuracy, quantified (ε,δ)-DP guarantee
The gradient inversion attack demonstrates that without DP, gradients leak significant information about the training data.